# Supply Chain Management with Uni (Rust)

BOM explosion, cost rollup, and supplier risk analysis using Uni's native Rust API.

In [ ]:
:dep uni-db = { path = "../../../crates/uni" }
:dep tokio = { version = "1", features = ["full"] }


In [ ]:
use uni_db::{Uni, DataType, IndexType, ScalarType, VectorMetric, VectorAlgo, VectorIndexCfg, Value};
use std::collections::HashMap;


In [ ]:
let db_path = "./supply_chain_db";

// Clean up any existing database
if std::path::Path::new(db_path).exists() {
    std::fs::remove_dir_all(db_path).unwrap();
}

let db = Uni::open(db_path).build().await?;
println!("Opened database at {}", db_path);


## 1. Define Schema

Parts (with name), Suppliers, and Products, along with ASSEMBLED_FROM and SUPPLIED_BY relationships.

In [ ]:
db.schema()
    .label("Part")
        .property("name", DataType::String)
        .property("sku",  DataType::String)
        .property("cost", DataType::Float64)
        .index("sku", IndexType::Scalar(ScalarType::Hash))
    .label("Supplier")
        .property("name", DataType::String)
    .label("Product")
        .property("name",  DataType::String)
        .property("price", DataType::Float64)
    .edge_type("ASSEMBLED_FROM", &["Product", "Part"], &["Part"])
    .edge_type("SUPPLIED_BY",    &["Part"],             &["Supplier"])
    .apply()
    .await?;

println!("Schema created successfully");

## 2. Ingest Data

7 parts, 3 suppliers, 2 products — same topology as the Python notebook.

In [ ]:
let session = db.session();
let tx = session.tx().await?;

// 7 parts
let part_props: Vec<HashMap<String, Value>> = vec![
    HashMap::from([("name".to_string(), Value::from("Resistor 10K")),    ("sku".to_string(), Value::from("RES-10K")),   ("cost".to_string(), Value::from(0.05))]),
    HashMap::from([("name".to_string(), Value::from("Capacitor 100uF")), ("sku".to_string(), Value::from("CAP-100UF")), ("cost".to_string(), Value::from(0.08))]),
    HashMap::from([("name".to_string(), Value::from("Motherboard X1")),  ("sku".to_string(), Value::from("MB-X1")),     ("cost".to_string(), Value::from(50.0))]),
    HashMap::from([("name".to_string(), Value::from("OLED Screen")),     ("sku".to_string(), Value::from("SCR-OLED")),  ("cost".to_string(), Value::from(30.0))]),
    HashMap::from([("name".to_string(), Value::from("Battery 4000mAh")), ("sku".to_string(), Value::from("BAT-4000")),  ("cost".to_string(), Value::from(15.0))]),
    HashMap::from([("name".to_string(), Value::from("ARM Processor")),   ("sku".to_string(), Value::from("PROC-ARM")),  ("cost".to_string(), Value::from(80.0))]),
    HashMap::from([("name".to_string(), Value::from("LCD Screen")),      ("sku".to_string(), Value::from("SCR-LCD")),   ("cost".to_string(), Value::from(20.0))]),
];

let part_vids = tx.bulk_insert_vertices("Part", part_props).await?;
let (res10k, cap100uf, mbx1, scr_oled, bat4000, proc_arm, scr_lcd) =
    (part_vids[0], part_vids[1], part_vids[2], part_vids[3], part_vids[4], part_vids[5], part_vids[6]);

// 3 suppliers
let sup_props: Vec<HashMap<String, Value>> = vec![
    HashMap::from([("name".to_string(), Value::from("ResistorWorld"))]),
    HashMap::from([("name".to_string(), Value::from("ScreenTech"))]),
    HashMap::from([("name".to_string(), Value::from("CoreComponents"))]),
];
let sup_vids = tx.bulk_insert_vertices("Supplier", sup_props).await?;
let (resistor_world, screen_tech, core_components) = (sup_vids[0], sup_vids[1], sup_vids[2]);

// 2 products
let prod_props: Vec<HashMap<String, Value>> = vec![
    HashMap::from([("name".to_string(), Value::from("Smartphone X")), ("price".to_string(), Value::from(599.0))]),
    HashMap::from([("name".to_string(), Value::from("TabletPro 10")), ("price".to_string(), Value::from(799.0))]),
];
let prod_vids = tx.bulk_insert_vertices("Product", prod_props).await?;
let (smartphone, tablet) = (prod_vids[0], prod_vids[1]);

// Smartphone X assembly
tx.bulk_insert_edges("ASSEMBLED_FROM", vec![
    (smartphone, mbx1,     HashMap::new()),
    (smartphone, scr_oled, HashMap::new()),
    (smartphone, bat4000,  HashMap::new()),
    (smartphone, proc_arm, HashMap::new()),
    (mbx1,       res10k,   HashMap::new()),
    (mbx1,       cap100uf, HashMap::new()),
]).await?;

// TabletPro 10 assembly
tx.bulk_insert_edges("ASSEMBLED_FROM", vec![
    (tablet, mbx1,    HashMap::new()),
    (tablet, scr_lcd, HashMap::new()),
    (tablet, bat4000, HashMap::new()),
    (tablet, proc_arm, HashMap::new()),
]).await?;

// Supply relationships
tx.bulk_insert_edges("SUPPLIED_BY", vec![
    (res10k,   resistor_world,  HashMap::new()),
    (cap100uf, resistor_world,  HashMap::new()),
    (scr_oled, screen_tech,     HashMap::new()),
    (scr_lcd,  screen_tech,     HashMap::new()),
    (mbx1,     core_components, HashMap::new()),
    (bat4000,  core_components, HashMap::new()),
    (proc_arm, core_components, HashMap::new()),
]).await?;

tx.commit().await?;
db.flush().await?;
println!("Data ingested and flushed");

## 3. BOM Explosion

Which products are affected if RES-10K is defective? Traverses the assembly hierarchy upward.

In [ ]:
let query = r#"
    MATCH (defective:Part {sku: 'RES-10K'})
    MATCH (product:Product)-[:ASSEMBLED_FROM*]->(defective)
    RETURN product.name AS name, product.price AS price
    ORDER BY product.price DESC
"#;

let results = session.query(query).await?;
println!("Products affected by defective RES-10K:");
for row in results.rows() {
    println!("  {:?}", row);
}
assert!(results.len() == 2, "Expected 2 affected products, got {}", results.len());

> **Bounded vs Unbounded Paths**: `[*]` writes no upper bound, so the planner
> supplies one — **100 hops**. That suits BOM explosion, where you want *every*
> affected product regardless of depth, as long as the hierarchy is shallower
> than 100: going past the bound returns a warning that results are incomplete. Use `[*1..5]`
> to cap traversal at a known depth, as shown in the queries below. Returning
> endpoints rather than a path variable also keeps these queries cheap — Uni
> only expands full paths when one is asked for.

## 4. Full BOM Listing

Every part in Smartphone X with its cost, ordered by cost descending.

In [ ]:
let query_parts = r#"
    MATCH (p:Product {name: 'Smartphone X'})-[:ASSEMBLED_FROM*1..5]->(part:Part)
    RETURN part.name AS part_name, part.sku AS sku, part.cost AS cost
    ORDER BY cost DESC
"#;

let results = session.query(query_parts).await?;
println!("Smartphone X BOM:");
for row in results.rows() {
    println!("  {:?}", row);
}

## 5. Cost Rollup

Total BOM cost per product — GROUP BY product with SUM of part costs.

In [ ]:
let query_rollup = r#"
    MATCH (p:Product)-[:ASSEMBLED_FROM*1..5]->(part:Part)
    RETURN p.name AS product, SUM(part.cost) AS total_bom_cost
    ORDER BY total_bom_cost DESC
"#;

let results = session.query(query_rollup).await?;
println!("BOM cost rollup per product:");
for row in results.rows() {
    println!("  {:?}", row);
}
assert!(results.len() == 2, "Expected 2 rows, got {}", results.len());

## 6. Supply Chain Risk

Which supplier is critical to the most products?

In [ ]:
let query_risk = r#"
    MATCH (p:Product)-[:ASSEMBLED_FROM*1..5]->(part:Part)-[:SUPPLIED_BY]->(s:Supplier)
    RETURN s.name AS supplier, COUNT(DISTINCT p) AS products_at_risk
    ORDER BY products_at_risk DESC
"#;

let results = session.query(query_risk).await?;
println!("Supplier risk analysis:");
for row in results.rows() {
    println!("  {:?}", row);
}
// CoreComponents supplies mbx1, bat4000, proc_arm — used by both products
println!("Top supplier: {:?}", results.rows()[0]);